<a href="https://colab.research.google.com/github/ClassNeuralNetwork/fire-detection/blob/main/yolo26n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install   ultralytics

In [ ]:
from ultralytics import YOLO
import yaml
import glob
import os
import torch
import matplotlib.pyplot as plt
import shap
from PIL import Image
from tensorflow.keras import layers, models

In [ ]:
import kagglehub


path = kagglehub.dataset_download("dataclusterlabs/fire-and-smoke-dataset")

print(path)

# Listar o conteúdo baixado
import os
print("Arquivos e pastas no diretório do dataset:", os.listdir(path))

In [ ]:
images = os.path.join(path, 'Datacluster Fire and Smoke Sample', 'Datacluster Fire and Smoke Sample')
annotations = os.path.join(path, 'Annotations', 'Annotations')
print(os.listdir(images))
print(os.listdir(annotations))

In [ ]:
base_dir = 'dataset'
for folder in ['train', 'val']:
  os.makedirs(os.path.join(base_dir, folder, 'images'), exist_ok=True)
  os.makedirs(os.path.join(base_dir, folder, 'labels'), exist_ok=True)
class_map = {"fire": 0, "smoke": 1}
images_list = []
annotations_list = []
for img in os.listdir(images):
  if img.endswith('.jpg'):
    images_list.append(img)
print(f'total de imagens: {len(images_list)}')
cut = int(0.8 * (len(images_list)))
train = images_list[:cut]
val = images_list[cut:]
print(f"""
imagens para treino:{len(train)}
imagens para validação:{len(val)}""")

In [ ]:
import shutil
for img in train:
  origem =os.path.join(images,img)
  destino = os.path.join(base_dir, 'train', 'images', img)
  shutil.copy(origem, destino)
for img in val:
  origem =os.path.join(images,img)
  destino = os.path.join(base_dir, 'val', 'images', img)
  shutil.copy(origem, destino)

In [ ]:
import xml.etree.ElementTree as ET
# !cat '/root/.cache/kagglehub/datasets/dataclusterlabs/fire-and-smoke-dataset/versions/3/Annotations/Annotations/Datacluster Fire and Smoke Sample (12).xml'
for xml in os.listdir(annotations):
  linhas_txt = []
  caminho = os.path.join(annotations, xml)
  tree = ET.parse(caminho)
  root = tree.getroot()


  obj = root.find('object')
  if obj is None:
        print(f"{xml}")
        continue
  name = obj.find('name').text



  bndbox = obj.find('bndbox')
  xmin = float(bndbox.find('xmin').text)
  ymin = float(bndbox.find('ymin').text)
  xmax = float(bndbox.find('xmax').text)
  ymax = float(bndbox.find('ymax').text)

  size = root.find('size')
  w = float(size.find('width').text)
  h = float(size.find('height').text)

  class_id = class_map.get(name)
  x_center = (xmin + xmax) / 2 / w
  y_center = (ymin + ymax) / 2 / h
  width = (xmax - xmin) / w
  height = (ymax - ymin) / h

  # print(f"""{xmin} + {xmax} / 2 / {w} = {x_center} """)
  # Annotation line format: <class_id> <x_center> <y_center> <width> <height> de https://labelformat.com/formats/object-detection/yolov11/
  # print(class_id, x_center, y_center, width, height)
  linhas_txt.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
  nome_base, _ = os.path.splitext(xml)
  nome_img = nome_base + '.jpg'
  if nome_img in train:
      current_split = 'train'
  elif nome_img in val:
      current_split = 'val'
  else:
      continue
  caminho_txt = os.path.join(base_dir, current_split, 'labels', nome_base + '.txt')
  with open(caminho_txt, 'w') as f:
      f.write('\n'.join(linhas_txt))

In [ ]:
data = """path: /content/dataset
train: /content/dataset/train/images  # Directory for training images
val: /content/dataset/val/images

names:
  0: fire
  1: smoke"""
caminho = os.path.join(base_dir, 'dataset.yaml')
with open(caminho, 'w') as f:
  f.write(data)

In [ ]:
!yolo task=detect mode=train model=yolo26n.pt data=/content/dataset/dataset.yaml epochs=100 imgsz=960

In [ ]:
custom_model = YOLO('runs/detect/train/weights/best.pt')

metrics = custom_model.val()

print(f"mAP50-90: {metrics.box.map: .4f}")
print(f"mAP50: {metrics.box.map50: .4f}")

In [ ]:
precision = metrics.box.mp
recall = metrics.box.mr
f1_score = metrics.box.f1.mean()

print(f"Precision: {precision: .4f}")
print(f"Recall: {recall: .4f}")
print(f"F1 Score: {f1_score: .4f}")

validated_classes = metrics.box.ap_class_index

for i, class_idx in enumerate(validated_classes):
    name = metrics.names[class_idx]
    p_class = metrics.box.p[i]
    r_class = metrics.box.r[i]
    print(f"Classe[{name}]: Precision = {p_class: .4f} | Recall = {r_class: .4f}")

In [ ]:
val_images_path = os.path.join(base_dir, 'val/images')
image_files = glob.glob(os.path.join(val_images_path, '*jpg')) + glob.glob(os.path.join(val_images_path, '*jpg'))

sample_images = image_files[:4]
results = custom_model(sample_images)

for i, r in enumerate(results):
    im_array = r.plot()
    im = Image.fromarray(im_array[..., ::-1])
    display(im)

In [ ]:
from google.colab import files
shutil.make_archive('treinos', 'zip', '/content/runs/')
files.download("treinos.zip")

In [ ]:
!pip install lime

In [ ]:
import numpy as np
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
from PIL import Image

# 1. Função wrapper de predição para o LIME
def yolo_predict_lime(images_np):
    preds = []
    for img_np in images_np:
        # O LIME passa imagens como floats ou uint8. Garantimos uint8 para o PIL/YOLO
        if img_np.max() <= 1.0:
            img_np = (img_np * 255).astype(np.uint8)
        else:
            img_np = img_np.astype(np.uint8)

        img = Image.fromarray(img_np)
        results = custom_model(img, verbose=False)

        # Inicializa probabilidades para as classes (0: fire, 1: smoke)
        prob_classes = np.zeros(len(custom_model.names))
        for r in results:
            if len(r.boxes) > 0:
                for box in r.boxes:
                    cls_id = int(box.cls[0].item())
                    conf = box.conf[0].item()
                    prob_classes[cls_id] = max(prob_classes[cls_id], conf)

        preds.append(prob_classes)
    return np.array(preds)

# 2. Inicializar o explicador LIME
explainer = lime_image.LimeImageExplainer()

train_images_path = os.path.join(base_dir, 'train/images')
train_image_files = glob.glob(os.path.join(train_images_path, '*.jpg'))

for idx, img_path in enumerate(train_image_files):
    # 1. Carregar e redimensionar a imagem
    img = Image.open(img_path).resize((224, 224))
    img_np = np.array(img)

    # 2. Gerar a explicação do LIME
    explanation = explainer.explain_instance(
        img_np,
        yolo_predict_lime,
        top_labels=2,
        hide_color=0,
        num_samples=100
    )

    # 3. Obter a máscara para a classe 'fire' (ID 0)
    temp, mask = explanation.get_image_and_mask(
        label=0,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    # 4. Plotar lado a lado para cada imagem
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))

    ax[0].imshow(img_np)
    ax[0].set_title(f"Imagem {idx+1} Original")
    ax[0].axis("off")

    ax[1].imshow(mark_boundaries(temp, mask))
    ax[1].set_title(f"Explicação LIME (Classe: fire)")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()